# 00 - Construcción del maestro de embalses

Este notebook construye la **tabla maestra de embalses** que se usará como referencia en el resto del proyecto.

## Objetivo

Consolidar en una única tabla los 35 embalses de la cuenca del Miño-Sil con datos SAIH, incorporando su capacidad máxima (necesaria para calcular el porcentaje de llenado, que es la variable objetivo del TFM).

## Fuentes utilizadas

- **Datos SAIH** (`data/raw/saih/Datos_Embalses.xlsx`): ID interno, nombre, coordenadas y sistema de explotación para cada embalse instrumentado con telemedida.
- **Inventario de embalses de la CHMS** (`Embalses_1_25000.shp` del Plan Hidrológico 2022-2027): nombre oficial, río, capacidad máxima (hm³), superficie y uso.

## Salida

`data/processed/maestro_embalses.parquet` y `.csv` — Tabla con 35 embalses y las columnas necesarias para el resto del pipeline.

## Notas

## 1. Imports y configuración

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Rutas del proyecto
DIR_RAW = Path("../data/raw")
DIR_EXTERNAL = Path("../data/external")
DIR_PROCESSED = Path("../data/processed")

# Rutas de archivos de entrada
PATH_SAIH_EMBALSES = DIR_RAW / "saih" / "Datos_Embalses.xlsx"
PATH_CHMS_EMBALSES = (
    DIR_EXTERNAL
    / "Datos-Cartograficos-Plan-Hidrologico-2022-2027"
    / "Hidrografia"
    / "Embalses_1_25000.shp"
)

# Ruta de salida
PATH_MAESTRO_PARQUET = DIR_PROCESSED / "maestro_embalses.parquet"
PATH_MAESTRO_CSV = DIR_PROCESSED / "maestro_embalses.csv"

## 2. Carga de embalses del SAIH

Se lee la hoja "Identificación Estaciones" del Excel de embalses. El archivo tiene cabecera multinivel (dos filas) porque las coordenadas X e Y se agrupan bajo un mismo encabezado.

In [2]:
embalses_saih = pd.read_excel(
    PATH_SAIH_EMBALSES,
    sheet_name="Identificación Estaciones",
    header=[0, 1],
)

# Aplanar cabeceras multinivel a nombres únicos
embalses_saih.columns = ["ID", "Nombre", "Municipio", "Provincia", "Sistema", "X", "Y"]

print(f"Embalses SAIH cargados: {len(embalses_saih)}")
embalses_saih.head()

Embalses SAIH cargados: 35


,ID,Nombre,Municipio,Provincia,Sistema,X,Y
0,E001,Belesar,O Saviñao,Lugo,Miño Alto,605857.785,4720461.028
1,E002,Os Peares,Carballedo,Lugo,Miño Alto,604739.725,4702113.693
2,E003,As Rozas,Villablino,León,Sil Superior,716279.933,4753843.899
3,E05A,Matalavilla - Presa,Páramo del Sil,León,Sil Superior,708038.359,4745675.320
4,E07A,Presa de Bárcena,Ponferrada,León,Sil Superior,700199.782,4716705.838


## 3. Carga del inventario de embalses de la CHMS

Se lee el shapefile `Embalses_1_25000.shp` de la Cartografía del Plan Hidrológico 2022-2027 de la CHMS. Contiene 58 embalses inventariados (más que los 35 del SAIH, ya que no todos los embalses de la demarcación están instrumentados con telemedida).

El sistema de referencia es **EPSG:25829 (ETRS89 UTM Zone 29N)**, coincidente con las coordenadas del SAIH.

In [3]:
embalses_chms = gpd.read_file(PATH_CHMS_EMBALSES)

print(f"Embalses CHMS cargados: {len(embalses_chms)}")
print(f"CRS: {embalses_chms.crs}")
embalses_chms[["NOMBRE", "RIO", "VOLUMEN", "SUP_EMBALS", "USO"]].head()

Embalses CHMS cargados: 58
CRS: EPSG:25829


,NOMBRE,RIO,VOLUMEN,SUP_EMBALS,USO
0,SANTIAGO,SIL,1.73,51.1906,H
1,PEÑADRADA,SIL,0.50,0.2308,H
2,MOURELA,MOURELA,0.03,0.0955,H
3,EDRADA CONSO,CONSO,0.20,3.0814,H
4,MAO,MAO,0.00,0.9824,NaN


## 4. Cruce por nombre normalizado

La estrategia principal de cruce es por **nombre normalizado**. Se aplica una normalización agresiva que:

- Pasa a mayúsculas y elimina tildes.
- Elimina sufijos frecuentes como "- Presa", "- C.H. ...".
- Elimina prefijos frecuentes como "Presa de", "Os", "As", "El", "La".
- Colapsa espacios repetidos.

Con esto se resuelven la mayor parte de las diferencias de nomenclatura entre SAIH y CHMS.

In [4]:
def normalizar_nombre(texto: str) -> str:
    """Normaliza un nombre de embalse para permitir el cruce entre fuentes.

    Elimina tildes, sufijos ('- Presa', '- C.H. ...'), prefijos ('Presa de',
    'Os', 'As', 'El', 'La') y colapsa espacios.
    """
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    # Quitar tildes
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    # Quitar sufijos comunes
    texto = re.sub(r"\s*-?\s*PRESA\s*$", "", texto)
    texto = re.sub(r"\s*-\s*C\.?H\.?\s+.*$", "", texto)
    # Quitar prefijos comunes
    texto = re.sub(r"^PRESA\s+DE\s+", "", texto)
    texto = re.sub(r"^(OS|AS|EL|LA)\s+", "", texto)
    # Colapsar espacios
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


# Aplicar normalización a ambas fuentes
embalses_saih["Nombre_norm"] = embalses_saih["Nombre"].apply(normalizar_nombre)
embalses_chms["Nombre_norm"] = embalses_chms["NOMBRE"].apply(normalizar_nombre)

# Cruce por nombre normalizado
cruce = embalses_saih.merge(
    embalses_chms[["NOMBRE", "Nombre_norm", "RIO", "VOLUMEN", "SUP_EMBALS", "USO"]],
    on="Nombre_norm",
    how="left",
)

n_casan = cruce["NOMBRE"].notna().sum()
print(f"Embalses cruzados por nombre: {n_casan} / {len(embalses_saih)}")

Embalses cruzados por nombre: 32 / 35


## 5. Casos manuales por proximidad espacial

Los embalses que no se resuelven por nombre se identifican por **proximidad espacial** al polígono más cercano del shapefile de la CHMS. Todos los casos manuales se resuelven con distancias inferiores a 100 metros.

| ID SAIH | Nombre SAIH | Nombre CHMS |
|---|---|---|
| E015 | Pías | PIAS O SAN AGUSTIN |
| E021 | Chandrexa | CHANDREJA DE QUEJIA |
| E025 | Leboreiro Mao | LEBOREIRO |

In [5]:
CASOS_MANUALES = {
    "E015": "PIAS O SAN AGUSTIN",
    "E021": "CHANDREJA DE QUEJIA",
    "E025": "LEBOREIRO",
}

for id_saih, nombre_chms in CASOS_MANUALES.items():
    fila_chms = embalses_chms[embalses_chms["NOMBRE"] == nombre_chms].iloc[0]
    mascara = cruce["ID"] == id_saih
    for col in ["NOMBRE", "RIO", "VOLUMEN", "SUP_EMBALS", "USO"]:
        cruce.loc[mascara, col] = fila_chms[col]

# Verificar que no queda ningún embalse sin capacidad
sin_asignar = cruce[cruce["VOLUMEN"].isna()]
assert len(sin_asignar) == 0, f"Quedan {len(sin_asignar)} embalses sin capacidad"
print(f"Todos los {len(cruce)} embalses tienen capacidad máxima asignada.")

Todos los 35 embalses tienen capacidad máxima asignada.


## 6. Consolidación del maestro y guardado

Se construye la tabla final con los nombres de columna del proyecto y se guarda en dos formatos:

- **Parquet**: para uso en el pipeline (rápido, tipos preservados).
- **CSV**: para inspección manual si es necesario.

In [6]:
maestro_embalses = cruce[
    ["ID", "Nombre", "X", "Y", "Sistema", "NOMBRE", "RIO", "VOLUMEN", "SUP_EMBALS", "USO"]
].copy()

maestro_embalses.columns = [
    "ID_SAIH",
    "Nombre_SAIH",
    "X",
    "Y",
    "Sistema",
    "Nombre_CHMS",
    "Rio",
    "Capacidad_hm3",
    "Superficie_ha",
    "Uso",
]

# Guardado
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)
maestro_embalses.to_parquet(PATH_MAESTRO_PARQUET, index=False)
maestro_embalses.to_csv(PATH_MAESTRO_CSV, index=False, encoding="utf-8-sig")

print(f"Maestro guardado:")
print(f"  - {PATH_MAESTRO_PARQUET} ({len(maestro_embalses)} embalses)")
print(f"  - {PATH_MAESTRO_CSV}")

maestro_embalses.head()

Maestro guardado:
  - ..\data\processed\maestro_embalses.parquet (35 embalses)
  - ..\data\processed\maestro_embalses.csv


,ID_SAIH,Nombre_SAIH,X,Y,Sistema,Nombre_CHMS,Rio,Capacidad_hm3,Superficie_ha,Uso
0,E001,Belesar,605857.785,4720461.028,Miño Alto,BELESAR,MIÑO,645.56,1724.3111,H
1,E002,Os Peares,604739.725,4702113.693,Miño Alto,PEARES,MIÑO,182.00,482.6588,H
2,E003,As Rozas,716279.933,4753843.899,Sil Superior,ROZAS,SIL,28.00,152.9916,H
3,E05A,Matalavilla - Presa,708038.359,4745675.320,Sil Superior,MATALAVILLA,VALSECO DE,65.04,183.8903,H
4,E07A,Presa de Bárcena,700199.782,4716705.838,Sil Superior,BARCENA,SIL,341.46,953.2600,I/S/H


## 7. Resumen del resultado

Balance final del cruce:

- **35 embalses** identificados con capacidad máxima verificada.
- **32 resueltos** por nombre normalizado.
- **3 resueltos** por proximidad espacial (verificación complementaria).
- **Cobertura: 100%.**

Salida disponible en `data/processed/maestro_embalses.parquet` para su uso en el resto del pipeline del TFM.